# 17 | LangGraph 记忆系统：让个人助理跨会话记住用户

前几篇已经分别讲过 `State`、`Reducer`、`thread_id`、`checkpoint` 和 Durable Execution。

这一篇把这些概念放到一个更贴近真实应用的问题里：**如果一个 AI 助理要长期服务同一个用户，它到底应该记住什么，怎么记，怎么验证真的记住了？**

这个实验要证明一件事：**记忆能力不是把完整聊天记录无限追加，而是把短期对话、长期用户画像和运行 checkpoint 分开管理。**

## 一、记忆能力到底有什么用

在一次性问答里，AI 只需要回答当前问题。但在真实应用里，用户经常希望 AI 能持续协作：

- 不要每次都重新介绍自己是谁；
- 不要反复说明自己的偏好、禁忌和工作背景；
- 不要忘记一个长期任务已经做到了哪一步；
- 不要把所有历史对话都塞进 prompt，增加成本和噪声；
- 不要把临时寒暄、模型猜测和真正有价值的用户信息混在一起。

所以记忆系统的价值不是“像人一样记住一切”，而是让 AI 应用保留对后续任务有价值的上下文：**用户是谁、偏好什么、项目处于什么状态、过去做过哪些决定。**

## 二、这个实验故意不依赖真实 LLM

很多记忆 demo 容易出现一个问题：模型从最近聊天历史里看到了信息，于是回答得像是“长期记住了”，但实际上系统并没有把长期记忆写入任何可靠存储。

为了让教学信号干净，这个 notebook 把重点放在 LangGraph 的状态流转和持久化设计上：

- 聊天回复使用一个确定性的 `compose_reply()`，方便观察 profile 是否真的参与了回答；
- 记忆抽取使用一个确定性的 `extract_memory_patch()`，避免模型 JSON 输出不稳定；
- 生产环境可以把这两个函数替换成真实 LLM 调用，尤其是用 structured output 做记忆抽取。

这样做不是为了否定 LLM，而是为了先把“记忆系统是否成立”验证清楚。

## 三、准备依赖

这个实验使用两类持久化：

- `SqliteSaver`：保存 LangGraph 的 checkpoint，也就是线程运行状态；
- 一个独立 SQLite 数据库：保存用户长期画像和记忆审计记录。

两者故意分开。checkpoint 适合恢复一次图运行和一条 thread 的状态；长期用户画像适合跨 thread、跨会话复用。分开数据库也能避免 checkpoint 写入和业务画像写入共用同一个 SQLite 连接。

In [ ]:
from __future__ import annotations

from importlib.metadata import version
from pathlib import Path
from typing import Annotated, Any, TypedDict
import json
import sqlite3
from datetime import datetime, timezone

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

print("langgraph", version("langgraph"))
print("langgraph-checkpoint-sqlite", version("langgraph-checkpoint-sqlite"))

## 四、长期记忆的数据边界

这里把用户记忆拆成两层：

- `profile`：当前可用的用户画像，例如姓名、城市、职业、偏好；
- `memory_events`：审计记录，记录每次为什么更新画像。

这样做有一个好处：应用回答问题时只需要读取干净的 `profile`，但排查问题时仍然能回看每一次更新来源。

In [ ]:
PROFILE_DB_PATH = Path("data/memory_profile_demo.sqlite")
CHECKPOINT_DB_PATH = Path("data/memory_checkpoint_demo.sqlite")
PROFILE_DB_PATH.parent.mkdir(parents=True, exist_ok=True)

# 为了让 notebook 每次运行都有一致结果，重建演示数据库。
for path in [PROFILE_DB_PATH, CHECKPOINT_DB_PATH]:
    if path.exists():
        path.unlink()

profile_db = sqlite3.connect(PROFILE_DB_PATH, check_same_thread=False)

profile_db.execute(
    """
    CREATE TABLE user_profiles (
        user_id TEXT PRIMARY KEY,
        profile_json TEXT NOT NULL,
        updated_at TEXT NOT NULL
    )
    """
)
profile_db.execute(
    """
    CREATE TABLE memory_events (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id TEXT NOT NULL,
        field TEXT NOT NULL,
        previous_value TEXT,
        new_value TEXT NOT NULL,
        source_text TEXT NOT NULL,
        created_at TEXT NOT NULL
    )
    """
)
profile_db.commit()


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def default_profile() -> dict[str, Any]:
    return {
        "name": None,
        "location": None,
        "occupation": None,
        "preferences": {},
        "facts": [],
    }


def load_profile_from_store(user_id: str) -> dict[str, Any]:
    row = profile_db.execute(
        "SELECT profile_json FROM user_profiles WHERE user_id = ?",
        (user_id,),
    ).fetchone()
    if row is None:
        return default_profile()
    return json.loads(row[0])


def save_profile_to_store(user_id: str, profile: dict[str, Any]) -> None:
    profile_db.execute(
        """
        INSERT INTO user_profiles(user_id, profile_json, updated_at)
        VALUES (?, ?, ?)
        ON CONFLICT(user_id) DO UPDATE SET
            profile_json = excluded.profile_json,
            updated_at = excluded.updated_at
        """,
        (user_id, json.dumps(profile, ensure_ascii=False), now_iso()),
    )
    profile_db.commit()


def append_memory_event(
    user_id: str,
    field: str,
    previous_value: Any,
    new_value: Any,
    source_text: str,
) -> None:
    profile_db.execute(
        """
        INSERT INTO memory_events(
            user_id, field, previous_value, new_value, source_text, created_at
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            user_id,
            field,
            json.dumps(previous_value, ensure_ascii=False),
            json.dumps(new_value, ensure_ascii=False),
            source_text,
            now_iso(),
        ),
    )
    profile_db.commit()

## 五、定义 State

`AssistantState` 里既有短期状态，也有长期记忆的临时载体：

- `messages`：当前 thread 的短期消息历史，使用 `add_messages` 追加；
- `user_id`：长期画像的查找键；
- `profile`：本轮从长期存储加载出来的用户画像；
- `memory_patch`：本轮从用户输入中提取出的画像增量；
- `turn_count`：观察 checkpoint 状态变化用。

注意：`profile` 不靠 `messages` 推断，它来自独立的 profile store。

In [ ]:
def increment(old: int | None, new: int) -> int:
    return (old or 0) + new


class AssistantState(TypedDict, total=False):
    messages: Annotated[list[BaseMessage], add_messages]
    user_id: str
    profile: dict[str, Any]
    memory_patch: dict[str, Any]
    turn_count: Annotated[int, increment]


def latest_human_text(messages: list[BaseMessage]) -> str:
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return str(message.content)
    return ""

## 六、节点一：加载长期画像

这个节点回答一个非常明确的问题：当前 `user_id` 过去有没有留下长期记忆？

如果换一个新的 `thread_id`，checkpoint 里的消息历史不会自动跟过去，但 `load_profile` 仍然可以通过同一个 `user_id` 读到长期画像。

In [ ]:
def load_profile(state: AssistantState) -> dict[str, Any]:
    user_id = state["user_id"]
    return {"profile": load_profile_from_store(user_id)}

## 七、节点二：根据画像生成回复

真实应用里，这里通常会调用 LLM，并把 `profile` 摘要放入 system prompt。

本实验用确定性函数代替 LLM，目的是让结果完全可验证：如果回复里使用了“美式咖啡”“不喝奶茶”等信息，那一定是从长期画像来的。

In [ ]:
def profile_summary(profile: dict[str, Any]) -> str:
    parts: list[str] = []
    if profile.get("name"):
        parts.append(f"姓名：{profile['name']}")
    if profile.get("location"):
        parts.append(f"城市：{profile['location']}")
    if profile.get("occupation"):
        parts.append(f"职业：{profile['occupation']}")

    preferences = profile.get("preferences", {})
    if preferences:
        parts.append("偏好：" + "；".join(f"{k}={v}" for k, v in preferences.items()))

    facts = profile.get("facts", [])
    if facts:
        parts.append("事实：" + "；".join(facts))

    return "；".join(parts) if parts else "暂无长期画像"


def compose_reply(user_text: str, profile: dict[str, Any]) -> str:
    name = profile.get("name") or "你"
    preferences = profile.get("preferences", {})
    drink = preferences.get("drink")
    avoid_drink = preferences.get("avoid_drink")

    if "推荐" in user_text and "饮品" in user_text:
        if drink:
            reply = f"{name}，我会优先推荐 {drink}。"
            if avoid_drink:
                reply += f"我也会避开你不喜欢的 {avoid_drink}。"
            return reply
        return "我还不知道你的饮品偏好，可以先从咖啡、茶或气泡水里选。"

    if "还记得" in user_text or "记得我" in user_text:
        return f"我当前记住的长期画像是：{profile_summary(profile)}。"

    if profile == default_profile():
        return "我们刚开始认识。我会只保存对后续有帮助的明确偏好和事实。"

    return f"收到。我会结合这些长期画像继续协助你：{profile_summary(profile)}。"


def chat(state: AssistantState) -> dict[str, Any]:
    messages = state.get("messages", [])
    profile = state.get("profile", default_profile())
    user_text = latest_human_text(messages)

    # 真实 LLM 应用里可以把这段作为 system prompt 的核心内容。
    _system_message = SystemMessage(
        content=(
            "你是一个有长期记忆的个人助理。"
            f"当前用户画像：{profile_summary(profile)}。"
            "只能使用明确保存的画像，不要从寒暄里猜测用户。"
        )
    )

    return {
        "messages": [AIMessage(content=compose_reply(user_text, profile))],
        "turn_count": 1,
    }

## 八、节点三：只从用户输入里提取候选记忆

记忆抽取最容易犯的错误，是把 assistant 的回复、模型推测、用户临时问题都写进长期记忆。

这里遵守三个规则：

- 只看最近一条用户消息；
- 只保存用户明确表达的信息；
- 不保存“我回来了”“你还记得吗”这类低价值信息。

生产环境可以把 `extract_memory_patch()` 换成 LLM structured output，但规则仍然应该保留。

In [ ]:
def extract_memory_patch(user_text: str) -> dict[str, Any]:
    patch: dict[str, Any] = {"preferences": {}, "facts": []}

    if "Alice" in user_text or "alice" in user_text.lower():
        patch["name"] = "Alice"
    if "北京" in user_text:
        patch["location"] = "北京"
    if "后端" in user_text:
        patch["occupation"] = "后端开发"
    elif "软件开发" in user_text:
        patch["occupation"] = "软件开发"

    if "不喝奶茶" in user_text or "不喜欢奶茶" in user_text:
        patch["preferences"]["avoid_drink"] = "奶茶"

    # 冲突更新：用户明确说“现在更喜欢拿铁”，应覆盖旧的 drink 偏好。
    if "现在更喜欢拿铁" in user_text or "喜欢拿铁" in user_text:
        patch["preferences"]["drink"] = "拿铁"
    elif "美式咖啡" in user_text:
        patch["preferences"]["drink"] = "美式咖啡"

    if "做后端开发" in user_text:
        patch["facts"].append("用户做后端开发")
    elif "做软件开发" in user_text:
        patch["facts"].append("用户做软件开发")

    return {key: value for key, value in patch.items() if value}


def extract_memory(state: AssistantState) -> dict[str, Any]:
    user_text = latest_human_text(state.get("messages", []))
    return {"memory_patch": extract_memory_patch(user_text)}

## 九、节点四：合并、去重、持久化

长期记忆不是简单追加。这里的合并策略是：

- `name/location/occupation`：有新值就覆盖旧值；
- `preferences`：按 key 更新，因此 `drink=拿铁` 会覆盖 `drink=美式咖啡`；
- `facts`：去重追加；
- 每次真实变化都写入 `memory_events`，方便审计。

这一步是记忆系统和普通聊天历史的关键区别。

In [ ]:
def merge_profile(
    user_id: str,
    profile: dict[str, Any],
    patch: dict[str, Any],
    source_text: str,
) -> dict[str, Any]:
    merged = json.loads(json.dumps(profile, ensure_ascii=False))

    for field in ["name", "location", "occupation"]:
        if field in patch and patch[field] != merged.get(field):
            append_memory_event(user_id, field, merged.get(field), patch[field], source_text)
            merged[field] = patch[field]

    for key, value in patch.get("preferences", {}).items():
        old_value = merged.setdefault("preferences", {}).get(key)
        if old_value != value:
            append_memory_event(user_id, f"preferences.{key}", old_value, value, source_text)
            merged["preferences"][key] = value

    facts = merged.setdefault("facts", [])
    for fact in patch.get("facts", []):
        if fact not in facts:
            append_memory_event(user_id, "facts", None, fact, source_text)
            facts.append(fact)

    return merged


def commit_memory(state: AssistantState) -> dict[str, Any]:
    user_id = state["user_id"]
    profile = state.get("profile", default_profile())
    patch = state.get("memory_patch", {})
    source_text = latest_human_text(state.get("messages", []))

    if not patch:
        return {"profile": profile}

    merged = merge_profile(user_id, profile, patch, source_text)
    save_profile_to_store(user_id, merged)
    return {"profile": merged}

## 十、构建图

图结构保持直观：

```text
START
  -> load_profile
  -> chat
  -> extract_memory
  -> commit_memory
  -> END
```

注意这个顺序：本轮回复使用的是“进入本轮之前已经保存的画像”；本轮用户新说出的信息会在回复之后写入长期画像，供下一轮或新会话使用。

In [ ]:
builder = StateGraph(AssistantState)
builder.add_node("load_profile", load_profile)
builder.add_node("chat", chat)
builder.add_node("extract_memory", extract_memory)
builder.add_node("commit_memory", commit_memory)

builder.add_edge(START, "load_profile")
builder.add_edge("load_profile", "chat")
builder.add_edge("chat", "extract_memory")
builder.add_edge("extract_memory", "commit_memory")
builder.add_edge("commit_memory", END)

checkpoint_db = sqlite3.connect(CHECKPOINT_DB_PATH, check_same_thread=False)
checkpointer = SqliteSaver(checkpoint_db)
graph = builder.compile(checkpointer=checkpointer)

print(graph.get_graph().draw_mermaid())

## 十一、辅助观察函数

实验中我们要同时观察三件事：

- 本轮助理回复；
- 长期画像是否更新；
- checkpoint 里当前 thread 的短期消息有多少。

In [ ]:
def print_profile(user_id: str) -> None:
    print(json.dumps(load_profile_from_store(user_id), ensure_ascii=False, indent=2))


def print_memory_events(user_id: str) -> None:
    rows = profile_db.execute(
        """
        SELECT field, previous_value, new_value, source_text
        FROM memory_events
        WHERE user_id = ?
        ORDER BY id
        """,
        (user_id,),
    ).fetchall()
    for field, previous_value, new_value, source_text in rows:
        print(f"- {field}: {previous_value} -> {new_value}｜来源：{source_text}")


def run_turn(user_id: str, thread_id: str, text: str) -> dict[str, Any]:
    config = {"configurable": {"thread_id": thread_id}}
    result = graph.invoke(
        {"user_id": user_id, "messages": [HumanMessage(content=text)]},
        config,
    )
    print(f"用户：{text}")
    print(f"助理：{result['messages'][-1].content}")
    print(f"thread={thread_id} 短期消息数：{len(result['messages'])}")
    return result

## 十二、实验 1：第一轮写入长期画像

用户明确说明姓名、城市、职业和饮品偏好。

这一轮回复不需要马上体现这些新信息，因为记忆是在 `chat` 之后的 `commit_memory` 节点写入的。我们重点观察长期画像是否被更新。

In [ ]:
USER_ID = "alice-001"

run_turn(
    user_id=USER_ID,
    thread_id="alice-session-1",
    text="你好，我叫 Alice，在北京做后端开发。我不喝奶茶，喜欢美式咖啡。",
)

print("\n长期画像：")
print_profile(USER_ID)

## 十三、实验 2：同一个 thread 继续对话

同一个 `thread_id` 会保留短期消息历史；同一个 `user_id` 会读取长期画像。

这里可以看到助理推荐饮品时，不需要用户重新说明偏好。

In [ ]:
run_turn(
    user_id=USER_ID,
    thread_id="alice-session-1",
    text="下午有点困，帮我推荐一个提神的饮品。",
)

## 十四、实验 3：换一个新 thread，但保留同一个用户

这是最关键的验证。

新 `thread_id` 代表一条新会话。它没有上一条 thread 的短期消息历史。如果助理仍然知道 Alice 喜欢美式咖啡、不喝奶茶，就说明信息来自长期画像，而不是来自聊天历史。

In [ ]:
new_session_result = run_turn(
    user_id=USER_ID,
    thread_id="alice-session-2",
    text="我明天还要早起，帮我推荐一个提神的饮品。",
)

print("\n新 thread 的消息内容：")
for message in new_session_result["messages"]:
    role = "用户" if isinstance(message, HumanMessage) else "助理"
    print(f"{role}: {message.content}")

## 十五、实验 4：处理冲突更新

长期记忆不能只会追加。如果用户明确说“现在更喜欢拿铁”，系统应该把 `drink` 偏好从“美式咖啡”更新为“拿铁”。

这也是结构化画像比纯文本记忆更可靠的地方。

In [ ]:
run_turn(
    user_id=USER_ID,
    thread_id="alice-session-3",
    text="我以前喜欢美式咖啡，但现在更喜欢拿铁。",
)

print("\n更新后的长期画像：")
print_profile(USER_ID)

print("\n再次推荐饮品：")
run_turn(
    user_id=USER_ID,
    thread_id="alice-session-4",
    text="帮我推荐一个提神的饮品。",
)

## 十六、查看记忆审计记录

一个严谨的记忆系统应该能回答：这条记忆从哪里来？什么时候写入？覆盖了什么旧值？

否则一旦记错，就很难排查。

In [ ]:
print_memory_events(USER_ID)

## 十七、checkpoint 记录了什么

长期画像保存在 profile store 里；LangGraph checkpoint 保存的是每条 thread 的运行状态。

这里观察 `alice-session-1` 的 checkpoint 数量。它能帮助我们恢复或检查这条 thread 的执行过程，但它不是跨所有会话的用户画像数据库。

In [ ]:
session_1_config = {"configurable": {"thread_id": "alice-session-1"}}
history = list(graph.get_state_history(session_1_config))

print(f"alice-session-1 checkpoint 数量：{len(history)}")
print("最新 checkpoint 的 state keys：", sorted(history[0].values.keys()))
print("最新 checkpoint 的短期消息数：", len(history[0].values.get("messages", [])))

## 十八、把确定性函数替换成真实 LLM

真实应用里，通常会替换两处：

第一，`chat` 节点调用 LLM。此时可以把用户画像作为 system prompt 的一部分：

```python
messages = [
    SystemMessage(content=f"用户画像：{profile_summary(profile)}"),
    *recent_messages,
]
response = llm.invoke(messages)
```

第二，`extract_memory` 节点用 structured output 抽取记忆。关键不是让模型自由发挥，而是让它返回固定 schema，并明确规则：

- 只提取用户明确说出的事实和偏好；
- 不要从 assistant 回复里提取；
- 不要保存寒暄和一次性请求；
- 遇到新旧冲突时返回“更新”而不是重复追加；
- 敏感信息默认不写入，除非业务确实需要且用户明确授权。

也就是说，LLM 可以负责理解自然语言，但记忆系统仍然应该由明确的数据结构、合并规则和审计机制约束。

## 十九、小结

这个实验里，LangGraph 的作用不是“自动拥有记忆”，而是提供一个清晰的状态流转框架：

- `State` 定义当前图运行需要携带什么；
- `add_messages` 管理短期消息历史；
- `SqliteSaver` 保存每条 thread 的 checkpoint；
- 独立 profile store 保存跨会话长期画像；
- 抽取、合并、审计节点把长期记忆变成可验证的数据流程。

因此，记忆能力的真正收益是：**让 AI 应用从“每次重新认识用户”，变成“能接着上次继续协作”，同时又不会把完整聊天历史、临时噪声和长期画像混成一团。**